# Example 4: Tool Calling（工具调用）

展示工具调用能力。

1.库与配置导入

In [7]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import math
import json

load_dotenv()

client = OpenAI(
    api_key=os.getenv("HY3_API_KEY"),
    base_url="https://tokenhub.tencentmaas.com/v1",
)

2.一次性工具调用（这里使用天气查询）

相关提问

In [8]:
# 一次工具调用
print("=== 一次工具调用示例 ===")
messages = [
    {"role": "user", "content": "北京今天天气怎么样？"},
]

=== 一次工具调用示例 ===


天气工具注册

In [9]:
weather_tool = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "获取指定城市的天气信息",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

使用

先确定工具调用

In [10]:
response = client.chat.completions.create(
    model="hy3",
    messages=messages,
    tools=weather_tool,
    tool_choice="auto",
)

print("=== 第一次响应 ===")
print("结束原因:", response.choices[0].finish_reason)

=== 第一次响应 ===
结束原因: tool_calls


回填调用工具得到的结果,再次和回填的结果一起构成上下文放入模型

In [11]:
if response.choices[0].finish_reason == "tool_calls":
    tool_call = response.choices[0].message.tool_calls[0]
    print("工具名称:", tool_call.function.name)
    print("工具参数:", tool_call.function.arguments)

    messages.append(response.choices[0].message)

    #回填结果
    city = json.loads(tool_call.function.arguments)["city"]
    weather_info = f"{city}今天天气晴朗，温度25-32°C，湿度60%。"
    print(f"\n模拟调用 get_weather('{city}') 返回:", weather_info)

    # 添加到消息，role使用tool
    messages.append({
        "role": "tool",
        "content": weather_info,
        "tool_call_id": tool_call.id
    })

    # 再次使用模型
    response = client.chat.completions.create(
        model="hy3",
        messages=messages,
        tools=weather_tool,
    )

    print("\n=== 最终回答 ===")
    print(response.choices[0].message.content)

工具名称: get_weather
工具参数: {"city": "北京"}

模拟调用 get_weather('北京') 返回: 北京今天天气晴朗，温度25-32°C，湿度60%。

=== 最终回答 ===
北京今天天气晴朗，气温在 25–32°C 之间，湿度约 60%，整体比较温暖舒适，注意防晒补水哦～


3.多轮工具循环调用

工具注册（这里使用圆和球体的面积计算）

In [12]:
# 多轮工具循环
def calculate_area(radius):
    return math.pi * radius ** 2

def calculate_volume(radius):
    return (4/3) * math.pi * radius ** 3

tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate_area",
            "description": "计算圆的面积",
            "parameters": {
                "type": "object",
                "properties": {
                    "radius": {
                        "type": "number",
                        "description": "圆的半径"
                    }
                },
                "required": ["radius"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_volume",
            "description": "计算球体的体积",
            "parameters": {
                "type": "object",
                "properties": {
                    "radius": {
                        "type": "number",
                        "description": "球体的半径"
                    }
                },
                "required": ["radius"]
            }
        }
    }
]

多轮循环，让模型判断工具什么时候调用什么时候结束

In [13]:
  print("\n=== 多轮工具循环示例 ===")
messages = [
    {"role": "user", "content": "一个半径为5的球体，它的表面积和体积分别是多少？"},
]

print("用户问题:", messages[0]["content"])
print()

max_iterations = 5
for i in range(max_iterations):
    response = client.chat.completions.create(
        model="hy3",
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )

    finish_reason = response.choices[0].finish_reason
    message = response.choices[0].message

    if finish_reason == "tool_calls":
        print(f"第 {i+1} 轮: 需要调用工具")
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)
            print(f"  - 调用: {tool_name}({tool_args})")

            if tool_name == "calculate_area":
                result = calculate_area(**tool_args)
            elif tool_name == "calculate_volume":
                result = calculate_volume(**tool_args)
            else:
                result = "未知工具"

            print(f"  - 返回: {result}")

            messages.append({
                "role": "tool",
                "content": str(result),
                "tool_call_id": tool_call.id
            })

        messages.append(message)

    elif finish_reason == "stop":
        print(f"\n第 {i+1} 轮: 完成回答")
        print("最终回答:", message.content)
        break

else:
    print("超过最大迭代次数")


=== 多轮工具循环示例 ===
用户问题: 一个半径为5的球体，它的表面积和体积分别是多少？

第 1 轮: 需要调用工具
  - 调用: calculate_volume({'radius': 5})
  - 返回: 523.5987755982989
第 2 轮: 需要调用工具
  - 调用: calculate_area({'radius': 5})
  - 返回: 78.53981633974483

第 3 轮: 完成回答
最终回答: <think:6124c78e></think:6124c78e>根据计算结果和球体几何公式：

**半径 r = 5 的球体**

1. **表面积**  
   公式：$S = 4\pi r^2 = 4 \times \pi \times 5^2 = 100\pi$  
   数值约为：**314.16**（平方单位）  
   *注：系统中暂无直接算球表面积的工具，此结果由圆面积公式 $A=\pi r^2$ 算出底圆面积 78.54 后乘以 4 得到，即 $4 \times 78.54 \approx 314.16$*

2. **体积**  
   调用球体体积工具计算结果为：**523.60**（立方单位）  
   （精确值 $\frac{4}{3}\pi \times 125 = \frac{500}{3}\pi \approx 523.5988$）

**总结**：表面积约 **314.16**，体积约 **523.60**。
